# Chapter 4. Drug–Drug Interaction: 학습 + Inference

이 노트북은 Chapter 4의 **모델 학습**과 **약물 상호작용 예측(inference)** 을 하나로 합친 버전입니다.

순서대로 셀을 실행하면 됩니다.


## 1. Setup

In [ ]:
%cd /content
!rm -rf ./dlfb-clone/
!git clone "https://github.com/deep-learning-for-biology/dlfb.git" dlfb-clone --branch main
%cd dlfb-clone

In [ ]:
%%bash
curl -LsSf https://astral.sh/uv/install.sh | sh && \
export PATH="/root/.local/bin:${PATH}" && \
uv pip compile ./requirements/{base,dlfb,graphs,gpu}.txt \
  --color never \
  --constraint ./requirements/constraints.txt | \
uv pip install -r - --system

In [ ]:
from google.colab import auth
auth.authenticate_user()
!dlfb-provision --chapter graphs

In [ ]:
%env JAX_DISABLE_JIT=False

try:
    import dlfb
except ImportError:
    import site
    site.main()
    import dlfb

import inspect
from dlfb.utils.display import display

## 2. 데이터 로드 및 탐색

In [ ]:
from ogb.linkproppred import LinkPropPredDataset
from dlfb.utils.context import assets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

dataset = LinkPropPredDataset(name="ogbl-ddi", root=assets("graphs/datasets"))
print(dataset.graph)
print(f'노드 수: {dataset.graph["num_nodes"]}, 엣지 수: {dataset.graph["edge_index"].shape[1]}')

In [ ]:
# 약물 이름 lookup 테이블 만들기
ddi_descriptions = pd.read_csv(
    assets("graphs/datasets/ogbl_ddi/mapping/ddi_description.csv.gz")
)
node_to_dbid_lookup = pd.read_csv(
    assets("graphs/datasets/ogbl_ddi/mapping/nodeidx2drugid.csv.gz")
)

first_drug = ddi_descriptions[["first drug id", "first drug name"]].rename(
    columns={"first drug id": "dbid", "first drug name": "drug_name"}
)
second_drug = ddi_descriptions[["second drug id", "second drug name"]].rename(
    columns={"second drug id": "dbid", "second drug name": "drug_name"}
)
dbid_to_name_lookup = (
    pd.concat([first_drug, second_drug]).drop_duplicates().reset_index(drop=True)
)
drugs_lookup = pd.merge(
    node_to_dbid_lookup.rename(columns={"drug id": "dbid", "node idx": "node_id"}),
    dbid_to_name_lookup,
    on="dbid",
    how="inner",
)
print(drugs_lookup.head())

## 3. 데이터셋 빌드

In [ ]:
import jax
from dlfb.graphs.dataset.builder import DatasetBuilder

node_limit = 500
rng = jax.random.PRNGKey(42)
rng, rng_dataset = jax.random.split(rng, 2)

dataset_splits = DatasetBuilder(path=assets("graphs/datasets")).build(
    node_limit, rng_dataset
)
print("데이터셋 빌드 완료!")
print(f"Train 노드 수: {dataset_splits['train'].n_nodes}")

## 4. 모델 학습

In [ ]:
import optax
from dlfb.graphs.model import DdiModel
from dlfb.graphs.train import train, auc_loss
from dlfb.graphs.inspect import plot_learning

rng, rng_init, rng_train = jax.random.split(rng, 3)

model = DdiModel(
    n_nodes=dataset_splits["train"].n_nodes,
    embedding_dim=512,
    dropout_rate=0.5,
    last_layer_self=True,
    degree_norm=False,
    n_mlp_layers=2,
)

state, metrics = train(
    state=model.create_train_state(
        rng=rng_init,
        dummy_input={
            "graph": dataset_splits["train"].graph,
            "pairs": dataset_splits["train"].pairs.get_dummy_input(),
        },
        tx=optax.adam(0.001),
    ),
    rng=rng_train,
    dataset_splits=dataset_splits,
    num_epochs=500,
    eval_every=1,
    loss_fn=auc_loss,
    norm_loss=True,
    store_path=assets("graphs/models/ddi_model"),
)

plot_learning(metrics)
print("학습 완료!")

## 5. Inference 함수 정의

In [ ]:
import jax.numpy as jnp
import numpy as np
from flax import linen as nn

def predict_interaction(state, graph, node_id_a, node_id_b):
    """두 약물 노드 간의 상호작용 확률 반환 (0~1)"""
    pair = jnp.array([[node_id_a, node_id_b]])
    score = state.apply_fn(
        {"params": state.params},
        graph, pair,
        is_training=False,
        is_pred=True,
    )
    return float(nn.sigmoid(score))


def predict_for_drug(state, graph, drug_name, drugs_lookup, top_k=10):
    """특정 약물과 상호작용 가능성 높은 약물 TOP K 반환. drug_name은 부분 매칭 지원."""
    match = drugs_lookup[drugs_lookup["drug_name"].str.contains(drug_name, case=False, na=False)]
    if match.empty:
        print(f"'{drug_name}' 약물을 찾을 수 없습니다.")
        print("사용 가능한 약물 목록 일부:", drugs_lookup["drug_name"].sample(10).tolist())
        return None

    query_node = int(match.iloc[0]["node_id"])
    query_drug_name = match.iloc[0]["drug_name"]
    print(f"'{query_drug_name}' (node_id={query_node}) 기준으로 예측 중...")

    all_node_ids = drugs_lookup["node_id"].values
    other_nodes = all_node_ids[all_node_ids != query_node]
    pairs = np.array([[query_node, n] for n in other_nodes])

    all_probs = []
    for start in range(0, len(pairs), 512):
        batch = jnp.array(pairs[start:start + 512])
        scores = state.apply_fn(
            {"params": state.params}, graph, batch,
            is_training=False, is_pred=True,
        )
        all_probs.append(np.array(nn.sigmoid(scores)))

    all_probs = np.concatenate(all_probs)
    name_lookup = drugs_lookup.set_index("node_id")["drug_name"].to_dict()

    results = pd.DataFrame({
        "query_drug": query_drug_name,
        "target_drug": [name_lookup.get(n, f"node_{n}") for n in other_nodes],
        "interaction_prob": all_probs,
    }).sort_values("interaction_prob", ascending=False).reset_index(drop=True)

    return results.head(top_k)


def predict_top_k_interactions(state, graph, node_ids, drugs_lookup, k=20, batch_size=512):
    """약물 집합 중 상호작용 확률 높은 상위 K 쌍 반환"""
    pairs = np.array([
        [node_ids[i], node_ids[j]]
        for i in range(len(node_ids))
        for j in range(i + 1, len(node_ids))
    ])
    print(f"총 {len(pairs):,}개 약물 쌍 예측 중...")

    all_probs = []
    for start in range(0, len(pairs), batch_size):
        batch = jnp.array(pairs[start:start + batch_size])
        scores = state.apply_fn(
            {"params": state.params}, graph, batch,
            is_training=False, is_pred=True,
        )
        all_probs.append(np.array(nn.sigmoid(scores)))

    all_probs = np.concatenate(all_probs)
    name_lookup = drugs_lookup.set_index("node_id")["drug_name"].to_dict()

    results = pd.DataFrame({
        "drug_a": [name_lookup.get(p[0], f"node_{p[0]}") for p in pairs],
        "drug_b": [name_lookup.get(p[1], f"node_{p[1]}") for p in pairs],
        "interaction_prob": all_probs,
    }).sort_values("interaction_prob", ascending=False).reset_index(drop=True)

    return results.head(k)


def check_known_vs_predicted(state, graph, dataset_splits, drugs_lookup, n_samples=10):
    """Test set의 실제 label vs 모델 예측 비교"""
    test_pos = dataset_splits["test"].pairs.pos[:n_samples]
    test_neg = dataset_splits["test"].pairs.neg[:n_samples]
    all_pairs = np.concatenate([test_pos, test_neg], axis=0)
    labels = np.array([1] * len(test_pos) + [0] * len(test_neg))

    scores = state.apply_fn(
        {"params": state.params}, graph,
        jnp.array(all_pairs),
        is_training=False, is_pred=True,
    )
    probs = np.array(nn.sigmoid(scores))
    name_lookup = drugs_lookup.set_index("node_id")["drug_name"].to_dict()

    results = pd.DataFrame({
        "drug_a": [name_lookup.get(p[0], f"node_{p[0]}") for p in all_pairs],
        "drug_b": [name_lookup.get(p[1], f"node_{p[1]}") for p in all_pairs],
        "true_label": labels,
        "predicted_prob": probs,
        "predicted_label": (probs >= 0.5).astype(int),
        "correct": ((probs >= 0.5).astype(int) == labels),
    })
    return results

print("Inference 함수 로드 완료!")

## 6. 예시 실행

In [ ]:
# ── 예시 1: 두 약물 간 상호작용 확률 ──
graph = dataset_splits["train"].graph
name_lookup = drugs_lookup.set_index("node_id")["drug_name"].to_dict()

node_a = int(dataset_splits["train"].pairs.pos[0, 0])
node_b = int(dataset_splits["train"].pairs.pos[0, 1])

prob = predict_interaction(state, graph, node_a, node_b)
print(f"{name_lookup.get(node_a, f'node_{node_a}')} ↔ {name_lookup.get(node_b, f'node_{node_b}')}")
print(f"예측 상호작용 확률: {prob:.4f}  →  {'✅ 상호작용 있음' if prob >= 0.5 else '❌ 상호작용 없음'}")

In [ ]:
# ── 예시 2: 특정 약물 기준 상호작용 가능성 TOP 10 ──
# drug_name 을 원하는 약물 이름으로 변경하세요 (부분 매칭 가능)
result_df = predict_for_drug(
    state, graph,
    drug_name="Warfarin",
    drugs_lookup=drugs_lookup,
    top_k=10,
)
if result_df is not None:
    display(result_df)

In [ ]:
# ── 예시 3: 랜덤 50개 약물 중 상호작용 확률 TOP 20 쌍 ──
sample_node_ids = drugs_lookup["node_id"].sample(50, random_state=42).tolist()
top_pairs = predict_top_k_interactions(
    state, graph,
    node_ids=sample_node_ids,
    drugs_lookup=drugs_lookup,
    k=20,
)
display(top_pairs)

In [ ]:
# ── 예시 4: 실제 label vs 모델 예측 비교 (test set) ──
comparison = check_known_vs_predicted(
    state, graph,
    dataset_splits=dataset_splits,
    drugs_lookup=drugs_lookup,
    n_samples=10,
)
display(comparison)
print(f"\n샘플 정확도: {comparison['correct'].mean():.2%}")